In [ ]:
import pandas as pd
import shap
import mlflow
from model_tuner import loadObjects
import sys
import os  # import operatings system for dir
import json
from pathlib import Path
from eda_toolkit import ensure_directory
import matplotlib.pyplot as plt
from model_metrics.model_registry import set_stores, stores, best_per_algo
import model_metrics.model_registry as mr
from model_metrics import (
    available,
    best_per_algo,
    load_best_per_algo,
    summarize_model_performance,
    show_calibration_curve,
    plot_threshold_metrics,
    show_roc_curve,
    show_pr_curve,
    show_confusion_matrix,
)

## Set Paths

In [ ]:
mlflow.set_tracking_uri(
    "/home/lshpaner/Python_Projects/bh_traffic/mlruns"
)

from model_metrics import (
    model_registry,
    set_stores,
    best_per_algo,
    load_best_per_algo,
    available,
)

model_registry.configure(root="/home/lshpaner/Python_Projects/bh_traffic")

In [ ]:
sys.path.insert(0, "../bh_traffic")

In [ ]:
data_path = "../data/processed"

In [ ]:
pred_dir = Path("../models/predictions/full_text_clean")

In [ ]:
base_path = os.path.join(os.pardir)

# Go up one level from 'notebooks' to parent directory,
# then into the 'data' folder
data_path = os.path.join(os.pardir, "data/processed")

# create image paths
image_path_png = os.path.join(base_path, "images", "png_images")
image_path_svg = os.path.join(base_path, "images", "svg_images")

# Use the function tob ensure'data' directory exists
ensure_directory(data_path)
ensure_directory(image_path_png)
ensure_directory(image_path_svg)

## Read in X, y

In [ ]:
X = pd.read_parquet("../data/processed/X.parquet")
y = pd.read_parquet("../data/processed/y.parquet").squeeze()

## Read in models from Mlflow

In [ ]:
set_stores("mlruns/models")

In [ ]:
best_per_algo(metric="valid Average Precision")
champs = load_best_per_algo(metric="valid Average Precision")

### Check available models in the registry

In [ ]:
available()

In [ ]:
mr.store_summary()

In [ ]:
best = best_per_algo(metric="valid Average Precision")
best

In [ ]:
lr_injury_or_fatal = champs["lr_injury_or_fatal"]
rf_injury_or_fatal = champs["rf_injury_or_fatal"]
xgb_injury_or_fatal = champs["xgb_injury_or_fatal"]
cat_injury_or_fatal = champs["cat_injury_or_fatal"]

### Confirm no outcome leakage exists

In [ ]:
lr_injury_or_fatal_feat_names = lr_injury_or_fatal.get_feature_names()

In [ ]:
[i for i in lr_injury_or_fatal_feat_names if "injury" in i or "fatal" in i]

## Set model titles, list models, and establish thresholds for performance assessment

In [ ]:
model_titles = [
    "Logistic Regression",
    "Random Forest",
    "XGBoost",
    "CatBoost",
]

models = [
    lr_injury_or_fatal,
    rf_injury_or_fatal,
    xgb_injury_or_fatal,
    cat_injury_or_fatal,
]

In [ ]:
thresholds = {
    "Logistic Regression": next(iter(lr_injury_or_fatal.threshold.values())),
    "Random Forest": next(iter(rf_injury_or_fatal.threshold.values())),
    "XGBoost": next(iter(xgb_injury_or_fatal.threshold.values())),
    "CatBoost": next(iter(cat_injury_or_fatal.threshold.values())),
}
thresholds

## Retrieve Test Set from LR models

In [ ]:
# per-model splits (each model has its own stored test indices)
X_train, y_train = lr_injury_or_fatal.get_train_data(X, y)
X_valid, y_valid = lr_injury_or_fatal.get_valid_data(X, y)
X_test, y_test = lr_injury_or_fatal.get_test_data(X, y)

In [ ]:
print(f"Training shape: {X_train.shape}")
print(f"Validation shape: {X_valid.shape}")
print(f"Test shape: {X_test.shape}")

In [ ]:
y_train.value_counts(1) * 100

In [ ]:
y_test.value_counts(1) * 100

## Performance Assessment Summary Table

In [ ]:
model_performance = summarize_model_performance(
    model=models,
    model_title=model_titles,
    X=X_test,
    y=y_test,
    model_type="classification",
    return_df=True,
    model_threshold=[thresholds[t] for t in model_titles],
)

model_performance

In [ ]:
len(xgb_injury_or_fatal.get_feature_names())

## Calibration and Threshold Selection (Validation Set)

### Calibration (Validation Set)

In [ ]:
show_calibration_curve(
    model=models,
    X=X_valid,
    y=y_valid,
    model_title=model_titles,
    bins=10,
    show_brier_score=True,
    overlay=True,
    linestyle_kwgs={"color": "black"},
    # figsize=(18, 10),
    tick_fontsize=14,
    label_fontsize=14,
)

### Threshold Selection (Validation Set)

In [ ]:
thresholds

In [ ]:
plot_threshold_metrics(
    model=models,
    model_title=model_titles,
    X_test=X_test,
    y_test=y_test,
    baseline_thresh=True,
    baseline_kwgs={
        "color": "red",
        "linestyle": "--",
        "linewidth": 1,
    },
    curve_kwgs={
        "linestyle": "-",
        "linewidth": 1,
    },
    threshold_kwgs={  # styling for the model-threshold vertical line
        "color": "blue",
        "linestyle": "--",
        "linewidth": 1,
    },
    text_wrap=40,
    subplots=True,
    figsize=(20, 8),
    model_threshold=[thresholds[t] for t in model_titles],
)

## AUROC Curves

In [ ]:
show_roc_curve(
    model=models,
    X=X_test,
    y=y_test,
    model_title=model_titles,
    decimal_places=3,
    n_cols=2,
    n_rows=1,
    curve_kwgs={
        "XGBoost": {"color": "blue", "linewidth": 1.5},
        "CatBoost": {"color": "black", "linewidth": 1.5},
    },
    linestyle_kwgs={"color": "gray", "linestyle": "--"},
    title="AUROC curves for all models",
    overlay=True,
)

## Precision-Recall Curves

In [ ]:
show_pr_curve(
    model=models,
    X=X_test,
    y=y_test,
    model_title=model_titles,
    decimal_places=2,
    n_cols=2,
    n_rows=1,
    curve_kwgs={
        "XGBoost": {"color": "blue", "linewidth": 1.5},
        "CatBoost": {"color": "black", "linewidth": 1.5},
    },
    title="Precision-Recall curves for all models",
    overlay=True,
    legend_loc="upper right",
)

## Confusion Matrices

In [ ]:
show_confusion_matrix(
    model=models,
    X=X_test,
    y=y_test,
    model_title=model_titles,
    cmap="Blues",
    text_wrap=32,
    subplots=True,
    n_cols=4,
    n_rows=1,
    figsize=(4, 4),
    label_fontsize=14,
    inner_fontsize=14,
    tick_fontsize=14,
    model_threshold=thresholds,
)

## SHAP

In [ ]:
champs

In [ ]:
import shap
import xgboost as xgb
import pandas as pd
import matplotlib.pyplot as plt

# ---------- recover real feature names ----------
calibrated = champs["xgb_injury_or_fatal"].estimator
pipeline = calibrated.estimator.estimator
booster = pipeline.named_steps["xgb"]
preprocessor = pipeline[:-1]
ct = pipeline.named_steps["preprocess_column_transformer_Preprocessor"]
ct_names = ct.get_feature_names_out()

rfe = pipeline.named_steps.get("feature_selection_RFE")
mask = rfe.support_ if rfe is not None else slice(None)
selected = [n.split("__")[-1] for n in ct_names[mask]]

# ---------- transform, forcing ndarray so columns= LABELS, not SELECTS ----------
X_trans_raw = preprocessor.transform(X_test)
if isinstance(X_trans_raw, pd.DataFrame):
    X_trans_raw = X_trans_raw.to_numpy()

assert X_trans_raw.shape[1] == len(selected), (X_trans_raw.shape, len(selected))

X_test_trans = pd.DataFrame(X_trans_raw, columns=selected, index=X_test.index)

# ---------- numeric frame for SHAP ----------
X_plot = X_test_trans.copy()
for col in X_plot.columns:
    if not pd.api.types.is_numeric_dtype(X_plot[col]):
        X_plot[col] = (
            X_plot[col].cat.codes
            if isinstance(X_plot[col].dtype, pd.CategoricalDtype)
            else pd.factorize(X_plot[col])[0]
        )
X_plot = X_plot.astype(float)

# ---------- SHAP (native XGBoost TreeSHAP) ----------
dmat = xgb.DMatrix(X_plot.to_numpy())
contribs = booster.get_booster().predict(
    dmat, pred_contribs=True, validate_features=False
)

shap_values = shap.Explanation(
    values=contribs[:, :-1],
    base_values=contribs[:, -1],
    data=X_plot.to_numpy(),
    feature_names=list(X_plot.columns),
)

# ---------- plot ----------
shap.summary_plot(shap_values, max_display=20, alpha=0.4, show=False)

plt.tight_layout()
plt.savefig("../images/shap_summary.png", dpi=300, bbox_inches="tight")
plt.show()